# USGS Geographic Shortcut Control — ATLAS

**Purpose:** Determine how much facility-size predictive signal is available from latitude and longitude alone. Uses the exact same 180-facility cohort and saved five geographic folds as the USGS terrain CNN. This is a diagnostic control, **not proof of shortcut learning**.

Run cells top to bottom in the ATLAS WSL kernel. This notebook does **not** retrain the CNN or handcrafted models and does **not** upload to S3.

**Evaluation caveat:** The same cohort has already been used to inspect previous experiments, so these comparisons are exploratory. Spatially disjoint groups do not guarantee no geographic autocorrelation.

## 1. Imports

Import the tabular models and metrics used for the location-only control, and fix the class names and seed. No terrain or image libraries are required in this notebook.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
LABELS = [0, 1, 2]
LABEL_NAMES = ["Small", "Medium", "Large"]

## 2. Locate repository and validate input paths

Find the ATLAS root from the working directory and check that the CNN predictions, fold manifest, registry, and handcrafted predictions all exist. The output folder is created only after those inputs are found.

In [2]:
# Notebook normally lives at ATLAS/models/. Find repository root without relying on Windows username.
start = Path.cwd().resolve()
candidates = [start, *start.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "models").is_dir() and (p / "outputs/usgs_cnn_spatial_v1_30epochs/oof_predictions.csv").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Cannot locate ATLAS repository from {start}. Open notebook from ATLAS/models/.")

CNN_PATH = PROJECT_ROOT / "outputs/usgs_cnn_spatial_v1_30epochs/oof_predictions.csv"
FOLDS_PATH = PROJECT_ROOT / "models/data/processed/usgs_naip_alignment/usgs_spatial_folds_v1.csv"
REGISTRY_CANDIDATES = [PROJECT_ROOT / "data/processed/im3_facility_registry.csv", PROJECT_ROOT / "models/data/processed/im3_facility_registry.csv"]
REGISTRY_PATH = next((p for p in REGISTRY_CANDIDATES if p.is_file()), None)
HANDCRAFTED_PATH = PROJECT_ROOT / "outputs/usgs_handcrafted_spatial_v1/oof_predictions.csv"
OUT_DIR = PROJECT_ROOT / "outputs/usgs_geographic_control_v1"

for label, path in {"CNN OOF": CNN_PATH, "Spatial folds": FOLDS_PATH, "Facility registry": REGISTRY_PATH, "Handcrafted OOF": HANDCRAFTED_PATH}.items():
    if path is None or not path.is_file():
        raise FileNotFoundError(f"Missing {label}: {path}")
    print(f"{label}: {path}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUT_DIR)

CNN OOF: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/outputs/usgs_cnn_spatial_v1_30epochs/oof_predictions.csv
Spatial folds: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/models/data/processed/usgs_naip_alignment/usgs_spatial_folds_v1.csv
Facility registry: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/data/processed/im3_facility_registry.csv
Handcrafted OOF: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/outputs/usgs_handcrafted_spatial_v1/oof_predictions.csv
Output directory: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/outputs/usgs_geographic_control_v1


## 3. Load cohort, coordinates, and fold manifest

Join CNN predictions, saved folds, and registry coordinates on facility id. Assertions confirm the 180 facilities, labels, and geographic groups match the CNN experiment before any model is fit.

In [3]:
cnn = pd.read_csv(CNN_PATH, dtype={"id": str})
folds = pd.read_csv(FOLDS_PATH, dtype={"id": str})
registry = pd.read_csv(REGISTRY_PATH, dtype={"id": str})
handcrafted = pd.read_csv(HANDCRAFTED_PATH, dtype={"id": str})
for frame in (cnn, folds, registry, handcrafted):
    frame["id"] = frame["id"].str.zfill(10)
print("Fold columns:", folds.columns.tolist())
print("Registry columns:", registry.columns.tolist())

required_folds = {"id", "fold", "size_class", "geo_group"}
required_registry = {"id", "lat", "lon"}
assert required_folds.issubset(folds.columns), f"Missing fold columns: {required_folds - set(folds.columns)}"
assert required_registry.issubset(registry.columns), f"Missing registry columns: {required_registry - set(registry.columns)}"
assert len(cnn) == 180 and cnn.id.is_unique
assert folds.id.is_unique and registry.id.is_unique

cohort = cnn[["id", "y_true", "fold"]].merge(
    folds[["id", "fold", "size_class", "geo_group"]], on="id", how="left",
    suffixes=("_cnn", "_manifest"), validate="one_to_one"
).merge(registry[["id", "lat", "lon"]], on="id", how="left", validate="one_to_one")
assert len(cohort) == 180 and cohort.id.is_unique
assert cohort[["fold_manifest", "size_class", "geo_group", "lat", "lon"]].notna().all().all()
assert (cohort.fold_cnn == cohort.fold_manifest).all(), "Fold assignments differ from CNN"
assert (cohort.y_true == cohort.size_class).all(), "Class labels differ from CNN"
assert cohort.groupby("geo_group").fold_cnn.nunique().max() == 1, "Geographic group crosses folds"
assert set(cohort.fold_cnn.astype(int)) == {1, 2, 3, 4, 5}
assert set(cohort.y_true.astype(int)) == {0, 1, 2}
assert np.isfinite(cohort[["lat", "lon"]].to_numpy(dtype=float)).all()
assert cohort.lat.between(-90, 90).all() and cohort.lon.between(-180, 180).all()
assert len(handcrafted) == 360, "Expected 180 OOF rows for each of two handcrafted models"
for name, sub in handcrafted.groupby("model"):
    assert len(sub) == 180 and sub.id.is_unique
    aligned = sub[["id", "fold", "y_true"]].merge(cohort[["id", "fold_cnn", "y_true"]], on="id", suffixes=("_hc", "_cohort"), validate="one_to_one")
    assert len(aligned) == 180
    assert (aligned.fold == aligned.fold_cnn).all() and (aligned.y_true_hc == aligned.y_true_cohort).all()

cohort = cohort.rename(columns={"fold_cnn": "fold"})[["id", "y_true", "fold", "geo_group", "lat", "lon"]]
print("Cohort:", len(cohort), "facilities; class counts:", cohort.y_true.value_counts().sort_index().to_dict())
print("Geographic folds:", cohort.fold.value_counts().sort_index().to_dict())
display(cohort.head())

Fold columns: ['id', 'lat', 'lon', 'size_class', 'geo_group', 'fold']
Registry columns: ['id', 'name', 'operator', 'state', 'state_abb', 'county', 'lat', 'lon', 'sqft', 'type']
Cohort: 180 facilities; class counts: {0: 7, 1: 50, 2: 123}
Geographic folds: {1: 36, 2: 37, 3: 36, 4: 35, 5: 36}


,id,y_true,fold,geo_group,lat,lon
0,0014930068,2,1,17_-54,35.049942,-106.542822
1,0014997588,2,1,18_-39,37.482763,-77.236455
2,0016282459,1,1,19_-39,39.140242,-76.606431
3,0031800424,1,1,16_-59,32.827866,-117.129934
4,0032560290,2,1,19_-61,38.650400,-121.491946


## 4. Define fixed geographic-only models

These models see **only** `lat` and `lon`. No elevation, DEM-derived features, square footage, state, or other variables are used. All preprocessing is fit separately on each outer training fold.

In [4]:
FEATURE_COLS = ["lat", "lon"]
models = {
    "Geo_LogisticRegression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
    ]),
    "Geo_RandomForest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(n_estimators=200, max_depth=4, min_samples_leaf=5,
                                              class_weight="balanced", random_state=SEED, n_jobs=-1))
    ])
}
print("Geographic-only features:", FEATURE_COLS)
print("Models:", list(models))

Geographic-only features: ['lat', 'lon']
Models: ['Geo_LogisticRegression', 'Geo_RandomForest']


## 5. Train five geographic folds and save held-out predictions

Fit each location-only model inside the saved geographic folds and predict only the held-out fold. Predictions and per-fold metrics are written so the comparison uses the same facilities as the CNN.

In [5]:
prediction_parts = []
fold_records = []
X = cohort[FEATURE_COLS]
y = cohort.y_true.astype(int)

for name, template in models.items():
    print("\n", name)
    for fold in sorted(cohort.fold.unique()):
        train = cohort.fold != fold
        test = cohort.fold == fold
        assert not (set(cohort.loc[train, "geo_group"]) & set(cohort.loc[test, "geo_group"]))
        model = clone(template)
        model.fit(X.loc[train], y.loc[train])
        preds = model.predict(X.loc[test]).astype(int)
        raw_probs = model.predict_proba(X.loc[test])
        classes = model.named_steps["classifier"].classes_
        probs = np.zeros((int(test.sum()), 3), dtype=float)
        for col, class_id in enumerate(classes):
            probs[:, int(class_id)] = raw_probs[:, col]
        part = cohort.loc[test, ["id", "y_true", "fold"]].reset_index(drop=True).copy()
        part["model"] = name
        part["y_pred"] = preds
        for class_id, label in enumerate(["small", "medium", "large"]):
            part[f"prob_{label}"] = probs[:, class_id]
        prediction_parts.append(part)
        metrics = {"model": name, "fold": int(fold), "n_test": int(test.sum()),
                   "accuracy": accuracy_score(y.loc[test], preds),
                   "balanced_accuracy": balanced_accuracy_score(y.loc[test], preds),
                   "macro_f1": f1_score(y.loc[test], preds, labels=LABELS, average="macro", zero_division=0)}
        fold_records.append(metrics)
        print(f"Fold {fold}: n={metrics['n_test']}, macro-F1={metrics['macro_f1']:.4f}, balanced accuracy={metrics['balanced_accuracy']:.4f}")

geo_oof = pd.concat(prediction_parts, ignore_index=True)
geo_fold_metrics = pd.DataFrame(fold_records)
for name, part in geo_oof.groupby("model"):
    assert len(part) == 180 and part.id.is_unique
    assert set(part.id) == set(cohort.id)
    p = part[["prob_small", "prob_medium", "prob_large"]].to_numpy()
    assert np.isfinite(p).all() and np.allclose(p.sum(axis=1), 1, atol=1e-5)
    assert np.array_equal(p.argmax(axis=1), part.y_pred.to_numpy())
geo_oof.to_csv(OUT_DIR / "oof_predictions.csv", index=False)
geo_fold_metrics.to_csv(OUT_DIR / "fold_metrics.csv", index=False)
print("Saved geographic control OOF predictions and fold metrics.")


 Geo_LogisticRegression
Fold 1: n=36, macro-F1=0.2357, balanced accuracy=0.3200
Fold 2: n=37, macro-F1=0.3163, balanced accuracy=0.2933
Fold 3: n=36, macro-F1=0.2694, balanced accuracy=0.2800
Fold 4: n=35, macro-F1=0.2843, balanced accuracy=0.2250
Fold 5: n=36, macro-F1=0.3355, balanced accuracy=0.2667

 Geo_RandomForest
Fold 1: n=36, macro-F1=0.2095, balanced accuracy=0.2200
Fold 2: n=37, macro-F1=0.2907, balanced accuracy=0.2867
Fold 3: n=36, macro-F1=0.2348, balanced accuracy=0.2533
Fold 4: n=35, macro-F1=0.2917, balanced accuracy=0.2250
Fold 5: n=36, macro-F1=0.2745, balanced accuracy=0.3111
Saved geographic control OOF predictions and fold metrics.


## 6. Compare all models on the same 180 held-out facilities

Score the majority baseline, the CNN, the handcrafted terrain models, and the geographic models with accuracy, balanced accuracy, and macro-F1. Sorting by macro-F1 shows whether latitude and longitude alone approach the terrain models.

In [6]:
comparison_records = []
def add_metrics(name, true, pred):
    true = np.asarray(true, dtype=int)
    pred = np.asarray(pred, dtype=int)
    comparison_records.append({
        "model": name,
        "n": len(true),
        "accuracy": accuracy_score(true, pred),
        "balanced_accuracy": balanced_accuracy_score(true, pred),
        "macro_f1": f1_score(true, pred, labels=LABELS, average="macro", zero_division=0),
        **{f"recall_{label}": float(np.mean(pred[true == i] == i)) for i, label in enumerate(["small", "medium", "large"])}
    })

add_metrics("Majority baseline", cnn.y_true, np.full(len(cnn), 2, dtype=int))
add_metrics("USGS CNN", cnn.y_true, cnn.y_pred)
for name, part in handcrafted.groupby("model"):
    add_metrics(f"Handcrafted {name}", part.y_true, part.y_pred)
for name, part in geo_oof.groupby("model"):
    add_metrics(name, part.y_true, part.y_pred)
comparison = pd.DataFrame(comparison_records).sort_values("macro_f1", ascending=False).reset_index(drop=True)
comparison.to_csv(OUT_DIR / "model_comparison.csv", index=False)
display(comparison.round(4))

,model,n,accuracy,balanced_accuracy,macro_f1,recall_small,recall_medium,recall_large
0,Handcrafted RandomForest,180,0.4000,0.3522,0.3060,0.2857,0.34,0.4309
1,Geo_LogisticRegression,180,0.3722,0.2765,0.2960,0.0000,0.48,0.3496
2,USGS CNN,180,0.6333,0.3208,0.2881,0.0000,0.06,0.9024
3,Majority baseline,180,0.6833,0.3333,0.2706,0.0000,0.00,1.0000
4,Geo_RandomForest,180,0.3444,0.2590,0.2674,0.0000,0.46,0.3171
5,Handcrafted LogisticRegression,180,0.3222,0.3222,0.2646,0.2857,0.38,0.3008


## 7. Classification reports and confusion matrices

Print a per-class report and confusion matrix for each geographic model. Predicted class counts show whether the location-only models collapse onto one size class.

In [7]:
for name, part in geo_oof.groupby("model"):
    print("\n" + "=" * 60 + "\n" + name)
    print(classification_report(part.y_true, part.y_pred, labels=LABELS, target_names=LABEL_NAMES, zero_division=0))
    display(pd.DataFrame(confusion_matrix(part.y_true, part.y_pred, labels=LABELS),
                         index=[f"Actual {s}" for s in LABEL_NAMES],
                         columns=[f"Pred {s}" for s in LABEL_NAMES]))
    print("Predicted class counts:", part.y_pred.value_counts().sort_index().to_dict())


Geo_LogisticRegression
              precision    recall  f1-score   support

       Small       0.00      0.00      0.00         7
      Medium       0.36      0.48      0.41        50
       Large       0.75      0.35      0.48       123

    accuracy                           0.37       180
   macro avg       0.37      0.28      0.30       180
weighted avg       0.61      0.37      0.44       180



,Pred Small,Pred Medium,Pred Large
Actual Small,0,3,4
Actual Medium,16,24,10
Actual Large,40,40,43


Predicted class counts: {0: 56, 1: 67, 2: 57}

Geo_RandomForest
              precision    recall  f1-score   support

       Small       0.00      0.00      0.00         7
      Medium       0.29      0.46      0.36        50
       Large       0.75      0.32      0.45       123

    accuracy                           0.34       180
   macro avg       0.35      0.26      0.27       180
weighted avg       0.59      0.34      0.40       180



,Pred Small,Pred Medium,Pred Large
Actual Small,0,5,2
Actual Medium,16,23,11
Actual Large,33,51,39


Predicted class counts: {0: 49, 1: 79, 2: 52}


## 8. Save experiment metadata and verify outputs

Write the control's purpose, features, and limitations beside the prediction files, then check that each expected artifact exists and is non-empty.

In [8]:
metadata = {
    "experiment": "usgs_geographic_control_v1",
    "purpose": "Latitude/longitude-only geographic shortcut diagnostic",
    "features": FEATURE_COLS,
    "n_facilities": len(cohort),
    "class_counts": {str(k): int(v) for k, v in cohort.y_true.value_counts().sort_index().items()},
    "fold_counts": {str(k): int(v) for k, v in cohort.fold.value_counts().sort_index().items()},
    "evaluation": "Five saved geographic-block-disjoint folds; pooled held-out predictions",
    "random_seed": SEED,
    "models": {name: model.get_params(deep=False).get("steps", []) and {"classifier": type(model.named_steps["classifier"]).__name__, "classifier_parameters": model.named_steps["classifier"].get_params()} for name, model in models.items()},
    "sources": {"cnn_oof": str(CNN_PATH), "spatial_folds": str(FOLDS_PATH),
                "registry": str(REGISTRY_PATH), "handcrafted_oof": str(HANDCRAFTED_PATH)},
    "limitations": ["Only 180 facilities, including seven Small", "No untouched independent test set",
                    "Exploratory model comparison", "Location-only predictive signal does not prove terrain CNN shortcut learning",
                    "Geographic blocks may still share regional characteristics"]
}
with open(OUT_DIR / "experiment_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
expected = ["oof_predictions.csv", "fold_metrics.csv", "model_comparison.csv", "experiment_metadata.json"]
for name in expected:
    path = OUT_DIR / name
    assert path.is_file() and path.stat().st_size > 0
    print(name, f"({path.stat().st_size:,} bytes)")
print("\nAll geographic-control artifacts saved to:", OUT_DIR)

oof_predictions.csv (34,416 bytes)
fold_metrics.csv (818 bytes)
model_comparison.csv (744 bytes)
experiment_metadata.json (2,516 bytes)

All geographic-control artifacts saved to: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/outputs/usgs_geographic_control_v1


**Audit CNN, handcrafted, and geographic outputs**

Check that all three experiment directories, CNN checkpoints, and prediction files are present. Ids, labels, folds, and class probabilities must agree across files before the scores are compared again.

In [12]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve ATLAS root from the models/ notebook directory
PROJECT_ROOT = Path.cwd().parent.resolve()

OUTPUTS = PROJECT_ROOT / "outputs"

CNN_DIR = OUTPUTS / "usgs_cnn_spatial_v1_30epochs"
TERRAIN_DIR = OUTPUTS / "usgs_handcrafted_spatial_v1"
GEO_DIR = OUTPUTS / "usgs_geographic_control_v1"

experiment_dirs = {
    "CNN": CNN_DIR,
    "Handcrafted": TERRAIN_DIR,
    "Geographic": GEO_DIR,
}

print("=== 1. EXPERIMENT DIRECTORIES ===")

for name, directory in experiment_dirs.items():
    print(f"\n{name}: {directory}")
    assert directory.is_dir(), f"Missing directory: {directory}"

    for file in sorted(directory.iterdir()):
        if file.is_file():
            print(f"  {file.name} ({file.stat().st_size:,} bytes)")

print("\n=== 2. CNN CHECKPOINTS ===")

for fold in range(1, 6):
    checkpoint = CNN_DIR / f"terrain_cnn_fold_{fold}.pt"

    assert checkpoint.is_file(), f"Missing: {checkpoint.name}"
    assert checkpoint.stat().st_size > 0

    print(f"Fold {fold}: OK")

print("\n=== 3. LOAD PREDICTIONS ===")

cnn = pd.read_csv(
    CNN_DIR / "oof_predictions.csv",
    dtype={"id": str},
)

terrain = pd.read_csv(
    TERRAIN_DIR / "oof_predictions.csv",
    dtype={"id": str},
)

geo = pd.read_csv(
    GEO_DIR / "oof_predictions.csv",
    dtype={"id": str},
)

for frame in [cnn, terrain, geo]:
    frame["id"] = frame["id"].str.zfill(10)

assert len(cnn) == 180
assert cnn["id"].is_unique

print("CNN rows:", len(cnn))
print("Handcrafted rows:", len(terrain))
print("Geographic rows:", len(geo))

print("\n=== 4. CHECK LABELS AND FOLDS ===")

reference = cnn[["id", "y_true", "fold"]].copy()

for name, frame in [
    ("Handcrafted", terrain),
    ("Geographic", geo),
]:
    assert {"id", "model", "y_true", "fold", "y_pred"}.issubset(
        frame.columns
    )

    for model_name, subset in frame.groupby("model"):
        assert len(subset) == 180, (
            f"{name}/{model_name}: expected 180 predictions"
        )
        assert subset["id"].is_unique

        merged = reference.merge(
            subset[["id", "y_true", "fold"]],
            on="id",
            how="outer",
            suffixes=("_cnn", "_other"),
            validate="one_to_one",
            indicator=True,
        )

        assert (merged["_merge"] == "both").all()
        assert (
            merged["y_true_cnn"] == merged["y_true_other"]
        ).all()
        assert (
            merged["fold_cnn"] == merged["fold_other"]
        ).all()

        print(f"{name}/{model_name}: IDs, labels, folds OK")

print("\n=== 5. CHECK PREDICTION VALUES ===")

for name, frame in [
    ("CNN", cnn),
    ("Handcrafted", terrain),
    ("Geographic", geo),
]:
    assert frame["y_true"].isin([0, 1, 2]).all()
    assert frame["y_pred"].isin([0, 1, 2]).all()
    assert frame["fold"].isin([1, 2, 3, 4, 5]).all()

    print(f"{name}: labels, predictions, folds OK")

print("\n=== 6. CHECK PROBABILITIES ===")

prob_columns = {
    "CNN": ["prob_small", "prob_medium", "prob_large"],
    "Handcrafted": ["prob_0", "prob_1", "prob_2"],
    "Geographic": ["prob_small", "prob_medium", "prob_large"],
}

for name, frame in [
    ("CNN", cnn),
    ("Handcrafted", terrain),
    ("Geographic", geo),
]:
    cols = prob_columns[name]

    missing = set(cols) - set(frame.columns)
    assert not missing, f"{name}: missing columns {missing}"

    probabilities = frame[cols].to_numpy(dtype=float)

    assert np.isfinite(probabilities).all(), (
        f"{name}: non-finite probabilities"
    )

    assert np.all(
        (probabilities >= 0) & (probabilities <= 1)
    ), f"{name}: probabilities outside [0, 1]"

    assert np.allclose(
        probabilities.sum(axis=1),
        1.0,
        atol=1e-4,
    ), f"{name}: probabilities do not sum to 1"

    assert np.array_equal(
        probabilities.argmax(axis=1),
        frame["y_pred"].to_numpy()
    ), f"{name}: predicted labels disagree with probabilities"

    print(f"{name}: probabilities OK")

print("\n=== AUDIT COMPLETE ===")
print("All prediction consistency checks passed.")
print("No files were modified or uploaded.")

=== 1. EXPERIMENT DIRECTORIES ===

CNN: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/outputs/usgs_cnn_spatial_v1_30epochs
  fold_metrics.csv (551 bytes)
  oof_predictions.csv (13,625 bytes)
  terrain_cnn_fold_1.pt (120,085 bytes)
  terrain_cnn_fold_2.pt (120,085 bytes)
  terrain_cnn_fold_3.pt (120,085 bytes)
  terrain_cnn_fold_4.pt (120,085 bytes)
  terrain_cnn_fold_5.pt (120,085 bytes)
  training_history.csv (6,774 bytes)

Handcrafted: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/outputs/usgs_handcrafted_spatial_v1
  experiment_metadata.json (1,182 bytes)
  fold_metrics.csv (849 bytes)
  model_comparison.csv (481 bytes)
  oof_predictions.csv (32,713 bytes)
  terrain_features.csv (37,375 bytes)

Geographic: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/outputs/usgs_geographic_control_v1
  experiment_metadata.json (2,516 bytes)
  fold_metrics.csv (818 bytes)
  model_comparison.csv (744 bytes)
  oof_predictions.csv (34,416 bytes)

=== 2. CNN CH

**Recompute metrics and match the saved table**

Independently score the CNN, handcrafted, and geographic predictions, then compare those numbers with the saved comparison within rounding error. This catches a mismatched id or label before the results are treated as final.

In [13]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

# Reuse cnn, terrain, and geo from the completed audit.
results = []

def evaluate(name, frame):
    y_true = frame["y_true"].to_numpy()
    y_pred = frame["y_pred"].to_numpy()

    results.append({
        "model": name,
        "n": len(frame),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0,
        ),
    })

evaluate("USGS CNN", cnn)

evaluate(
    "Majority baseline",
    cnn.assign(y_pred=2),
)

for model_name, subset in terrain.groupby("model"):
    evaluate(f"Handcrafted/{model_name}", subset)

for model_name, subset in geo.groupby("model"):
    evaluate(f"Geographic/{model_name}", subset)

verified_metrics = pd.DataFrame(results)

assert len(verified_metrics) == 6
assert (verified_metrics["n"] == 180).all()

display(
    verified_metrics.sort_values(
        "macro_f1", ascending=False
    ).round(4)
)

print("\n=== SAVED METRIC CONSISTENCY ===")

# Compare reported pooled results with independently
# recomputed predictions, allowing only rounding differences.
expected = {
    "USGS CNN": (0.6333, 0.3208, 0.2881),
    "Majority baseline": (0.6833, 0.3333, 0.2706),
    "Handcrafted/LogisticRegression": (0.3222, 0.3222, 0.2646),
    "Handcrafted/RandomForest": (0.4000, 0.3522, 0.3060),
    "Geographic/Geo_LogisticRegression": (0.3722, 0.2765, 0.2960),
    "Geographic/Geo_RandomForest": (0.3444, 0.2590, 0.2674),
}

for row in verified_metrics.itertuples(index=False):
    actual = np.array([
        row.accuracy,
        row.balanced_accuracy,
        row.macro_f1,
    ])

    target = np.array(expected[row.model])

    assert np.allclose(actual, target, atol=0.0001), (
        f"Metric mismatch: {row.model}\n"
        f"Actual: {actual}\nExpected: {target}"
    )

    print(f"{row.model}: OK")

print("\nAll six model evaluations verified.")

,model,n,accuracy,balanced_accuracy,macro_f1
3,Handcrafted/RandomForest,180,0.4000,0.3522,0.3060
4,Geographic/Geo_LogisticRegression,180,0.3722,0.2765,0.2960
0,USGS CNN,180,0.6333,0.3208,0.2881
1,Majority baseline,180,0.6833,0.3333,0.2706
5,Geographic/Geo_RandomForest,180,0.3444,0.2590,0.2674
2,Handcrafted/LogisticRegression,180,0.3222,0.3222,0.2646



=== SAVED METRIC CONSISTENCY ===
USGS CNN: OK
Majority baseline: OK
Handcrafted/LogisticRegression: OK
Handcrafted/RandomForest: OK
Geographic/Geo_LogisticRegression: OK
Geographic/Geo_RandomForest: OK

All six model evaluations verified.


**Preview the geographic prediction file**

Open the saved geographic out-of-fold table and show its columns, model counts, and first rows. Use this to confirm both location-only models were written.

In [10]:
import pandas as pd

geo_path = (
    PROJECT_ROOT
    / "outputs/usgs_geographic_control_v1/oof_predictions.csv"
)

geo_check = pd.read_csv(geo_path, dtype={"id": str})

print("Geographic prediction columns:")
print(geo_check.columns.tolist())

print("\nAvailable models:")
print(geo_check["model"].value_counts())

print("\nFirst five rows:")
display(geo_check.head())

Geographic prediction columns:
['id', 'y_true', 'fold', 'model', 'y_pred', 'prob_small', 'prob_medium', 'prob_large']

Available models:
model
Geo_LogisticRegression    180
Geo_RandomForest          180
Name: count, dtype: int64

First five rows:


,id,y_true,fold,model,y_pred,prob_small,prob_medium,prob_large
0,0014930068,2,1,Geo_LogisticRegression,2,0.327612,0.236787,0.435601
1,0014997588,2,1,Geo_LogisticRegression,1,0.368946,0.375255,0.255799
2,0016282459,1,1,Geo_LogisticRegression,1,0.352675,0.403261,0.244065
3,0031800424,1,1,Geo_LogisticRegression,2,0.307877,0.179431,0.512691
4,0032560290,2,1,Geo_LogisticRegression,2,0.265394,0.224364,0.510242


## Interpretation and next checkpoint

Compare the geographic-only models with the CNN and handcrafted baselines. Similar performance suggests geography alone may explain some predictive signal, but **does not prove** the CNN learned a geographic shortcut. The control may also struggle because the held-out blocks differ geographically from training blocks.

**Next:** review the comparison and confusion matrices, then validate the three experiment output directories before publishing to S3. Do not automatically select or tune models based on these held-out results.